# 03 &mdash; Node classification on Cora

> **Reading order.** These notebooks are a course, not a reference. Each one
> assumes the previous. `01` builds the tensor vocabulary, `02` builds the layers,
> `03` trains them on Cora, `04` proves they are correct and breaks them
> deliberately, `05` moves to graph-level tasks.
>
> Everything imports the local `gnn_lab` package -- edit the package and re-run
> the cell. The self-contained Kaggle run lives in `kaggle/`.

**Goal.** Train the three architectures on a real dataset under a fair protocol,
and learn to read the results critically rather than just reporting a number.

Cora is the "MNIST of graph learning": 2708 machine-learning papers, 1433
bag-of-words features, 7 classes, 5278 undirected citation edges. Small enough
to train in seconds on CPU, real enough that every published GNN number you will
compare against was produced on it.

---

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))     # so `import gnn_lab` works from notebooks/

import torch
import torch.nn as nn
import torch.nn.functional as F

from gnn_lab import (GCNScratch, GraphSAGEScratch, GATScratch, GINScratch,
                     GCNConvScratch, SAGEConvScratch, GATConvScratch, GINConvScratch,
                     load_cora, load_mutag, triangle_square_dataset, karate_club,
                     toy_graph, path_graph, two_communities,
                     train_node, train_node_multi_seed, train_graph_classification,
                     evaluate_node, set_seed, get_device, count_parameters,
                     accuracy, macro_f1, majority_baseline,
                     mean_pairwise_cosine, dirichlet_energy)
from gnn_lab.ops import (scatter_add, scatter_mean, scatter_max, scatter_softmax,
                         gcn_norm, add_self_loops, remove_self_loops, to_undirected,
                         degree, dense_to_edge_index)
from gnn_lab.layers.pooling import collate_graphs, global_pool, graph_batches
from gnn_lab import equivalence, viz

DEVICE = get_device()
RESULTS = os.path.join("..", "results")
os.makedirs(RESULTS, exist_ok=True)

print("torch  :", torch.__version__)
print("device :", DEVICE)
print("PyG    :", "available" if equivalence.PYG_AVAILABLE else f"missing ({equivalence.PYG_IMPORT_ERROR})")
set_seed(0)


## 1. Two loaders for the same graph

The repo ships two Cora loaders, and having both is deliberate.

**`source="planetoid"`** reads the canonical pickle files straight from GitHub
(`kimiyoung/planetoid`). That is the exact byte stream behind the 81.5% GCN
number in the original paper, and the exact thing
`torch_geometric.datasets.Planetoid` reads -- which is what makes the numerical
comparison in notebook 04 meaningful.

**`source="linqs"`** reads the official plain-text release from the LINQS group
at UC Santa Cruz: one line per paper with 1433 `0`/`1` columns, plus a separate
edge list. No scipy. Use this when you want to actually *look* at the data.

They describe the same graph but do **not** share a node ordering, so results
from one cannot be compared node-for-node with the other.

In [ ]:
cora = load_cora(source="planetoid", row_normalize=False)
print()
cora_text = load_cora(source="linqs", row_normalize=False)
print()
print("same graph, two encodings:")
print(f"  planetoid: {cora.num_nodes} nodes, {cora.num_edges} directed edges, {cora.num_classes} classes")
print(f"  linqs    : {cora_text.num_nodes} nodes, {cora_text.num_edges} directed edges, {cora_text.num_classes} classes")
print(f"  identical class sizes: "
      f"{torch.equal(torch.bincount(cora.y.cpu(), minlength=7).sort().values, torch.bincount(cora_text.y.cpu(), minlength=7).sort().values)}")
print()
print("linqs exposes the class names because the text file has them:")
print(" ", cora_text.class_names)
print("  sizes:", torch.bincount(cora_text.y.cpu(), minlength=7).tolist())
print()
print("planetoid does not: the pickle stores one-hot columns with no name mapping,")
print("  so cora.class_names is", cora.class_names, "-- the index is all you get.")

## 2. Two gotchas in the raw files

Both were found by inspecting the actual bytes rather than trusting a README.
Both silently change your results if you get them wrong.

### Gotcha 1: `test.index` is not sorted

The file lists 1000 node indices in the range `[1708, 2707]` -- but **not in
order**. It starts `2692, 2532, 2050, 1715, 2362, ...`.

Row `k` of the test feature block `tx` belongs to node `sorted(test_index)[k]`,
so naively stacking `allx` and `tx` puts the test features in the wrong order.
You must un-permute:

```python
x[test_index] = x[sorted_test_index]
```

Skip it and you still train happily -- you just evaluate on a silently
relabelled test set.

In [ ]:
import os
root = os.path.join("..", "data", "cora_planetoid")
path = os.path.join(root, "ind.cora.test.index")
idx = [int(l) for l in open(path).read().split()] if os.path.exists(path) else None

if idx is None:
    print("(raw files not cached yet -- run the loader cell above first)")
else:
    print(f"test.index has {len(idx)} entries")
    print(f"  first 8 as stored : {idx[:8]}")
    print(f"  sorted            : {sorted(idx)[:8]} ...")
    print(f"  already sorted?   : {idx == sorted(idx)}")
    print(f"  range             : [{min(idx)}, {max(idx)}]  (contiguous: {sorted(idx) == list(range(min(idx), max(idx)+1))})")
    print()
    print("Because the range is contiguous, `sorted(test_index)` is just 1708..2707,")
    print("and the un-permutation reassigns each stored row to its true node id.")

### Gotcha 2: the adjacency dict contains duplicate entries

Cora's `graph` pickle has **10858** adjacency entries. The true graph has
**5278** undirected edges, i.e. **10556** directed. So 302 entries are
duplicates -- 151 reciprocal pairs stored in both directions *twice*.

PyTorch Geometric removes them and strips self-loops. The original `tkipf/gcn`
code does not. PyG even carries a source comment about it:

> `# NOTE: There are some duplicated edges and self loops in the datasets.`
> `#       Other implementations do not remove them!`

This matters because duplicate edges **inflate node degrees**, and degree is
exactly what GCN's normalisation divides by. `gnn_lab` defaults to PyG's
behaviour (`dedupe=True`) so comparisons are fair, and keeps the switch so you
can measure what the difference is worth -- notebook 04 does exactly that.

In [ ]:
for dedupe in (True, False):
    d = load_cora(source="planetoid", row_normalize=False, dedupe=dedupe, verbose=False)
    deg = degree(d.edge_index[1], d.num_nodes)
    print(f"dedupe={dedupe!s:5}  edges {d.num_edges:5d}   "
          f"degree mean {float(deg.mean()):.4f}  max {int(deg.max())}")
print()
print("difference:", 10858 - 10556, "stray adjacency entries")
print("Same underlying graph -- the LINQS text file has 5429 lines that also")
print("collapse to 5278 unique undirected edges, for the same reason.")

## 3. Semi-supervised means the unlabelled nodes still matter

The split is 140 train / 500 val / 1000 test -- only **5.2% of nodes are ever
labelled for training**. But the loss is *masked*, not the graph: all 2708 nodes
participate in every forward pass, so information flows through the unlabelled
nodes and shapes every prediction.

That is the actual idea behind GNNs for semi-supervised learning, and it is easy
to miss if you only look at the training loop.

In [ ]:
print(f"nodes           : {cora.num_nodes}")
print(f"  train         : {int(cora.train_mask.sum()):5d}  ({cora.train_mask.float().mean()*100:.1f}%)")
print(f"  val           : {int(cora.val_mask.sum()):5d}  ({cora.val_mask.float().mean()*100:.1f}%)")
print(f"  test          : {int(cora.test_mask.sum()):5d}  ({cora.test_mask.float().mean()*100:.1f}%)")
print(f"  unlabelled    : {cora.num_nodes - int((cora.train_mask|cora.val_mask|cora.test_mask).sum())}")
print()
print("splits disjoint:",
      not bool(((cora.train_mask & cora.val_mask) | (cora.val_mask & cora.test_mask)
                | (cora.train_mask & cora.test_mask)).any()))
print("splits cover every node:",
      bool((cora.train_mask | cora.val_mask | cora.test_mask).all()))
print()
print("labels per class in the TRAIN split (the Planetoid protocol is 20 per class):")
print(" ", torch.bincount(cora.y[cora.train_mask], minlength=7).tolist())
print("labels per class in the TEST split (heavily imbalanced):")
print(" ", torch.bincount(cora.y[cora.test_mask], minlength=7).tolist())

### Homophily: the assumption everything rests on

Message passing assumes a node's neighbours are informative about its label. On
Cora that holds strongly -- citing papers usually share a topic. It is worth
measuring rather than assuming, because on heterophilous graphs (fraud,
dating, some protein networks) the assumption inverts and plain GCN actively
hurts.

In [ ]:
y_src, y_dst = cora.y[cora.edge_index[0]], cora.y[cora.edge_index[1]]
h = (y_src == y_dst).float().mean()
print(f"edge homophily on Cora: {float(h):.4f}")
print(f"  -> {float(h)*100:.1f}% of citation edges join papers of the same class")
print()
k = karate_club()
hk = (k["y"][k["edge_index"][0]] == k["y"][k["edge_index"][1]]).float().mean()
print(f"for reference, the karate club: {float(hk):.4f}")
print()
print("A random edge would match by chance with probability sum(p_c^2) =", 
      f"{float((torch.bincount(cora.y.cpu(), minlength=7).float()/cora.num_nodes).pow(2).sum()):.4f}")
print("so 0.81 is far above chance -- but it is not 1.0, and that gap is why")
print("learned attention (GAT) can add anything at all.")

In [ ]:
viz.draw_graph(cora.edge_index[:, ::3].cpu(), labels=cora.y.cpu(), node_size=4,
               title=f"Cora ({cora.num_nodes} papers, coloured by class)",
               path=f"{RESULTS}/03_cora.png")

## 4. Training protocol

One shared protocol for all three models so the comparison means something:

* 2 layers, 16 hidden channels
* Adam, lr `0.01`, weight decay `5e-4`
* dropout `0.5` (GAT uses `0.6` on its input, per the paper)
* 200 epochs, **full-batch** -- all 2708 nodes every epoch
* 3 seeds, reported as mean +/- std

**Reporting note that matters.** We report test accuracy *at the epoch where
validation accuracy peaked*, not the best test accuracy ever observed. Taking
the max over epochs is test-set selection and inflates results by a point or
two. It is common in GNN papers and it is wrong. `train_node` records both so
you can see the gap for yourself.

In [ ]:
def build(name):
    f, h, c = cora.num_features, 16, cora.num_classes
    return {"gcn":  lambda: GCNScratch(f, h, c, num_layers=2, dropout=0.5),
            "sage": lambda: GraphSAGEScratch(f, h, c, num_layers=2, dropout=0.5, aggr="mean"),
            "gat":  lambda: GATScratch(f, h, c, heads=8, dropout=0.6)}[name]

set_seed(0)
model = build("gcn")().to(DEVICE)
res = train_node(model, cora.to(DEVICE), model_name="GCN", epochs=200,
                 lr=1e-2, weight_decay=5e-4, device=DEVICE, log_every=20)

print()
print(f"test acc at best val epoch : {res.test_acc_at_best_val*100:.2f}%   <- report this")
print(f"best test acc over epochs  : {res.best_test_acc*100:.2f}%   <- test-set selection, inflated")
print(f"gap                        : {(res.best_test_acc - res.test_acc_at_best_val)*100:+.2f} points")
print(f"best epoch                 : {res.extra['best_epoch']}")
print(f"training time              : {res.train_time_s:.1f}s")

In [ ]:
viz.plot_training_curves({"GCN": res.history}, title="GCN on Cora, 200 epochs",
                         path=f"{RESULTS}/03_curves_gcn.png")

## 5. All three architectures, multiple seeds

A single Cora run has ~1% seed-to-seed variance. Comparing two architectures on
one seed each is therefore close to meaningless -- three seeds is the minimum
for a claim, and even that is optimistic.

In [ ]:
SEEDS = [0, 1, 2]
summary, histories = {}, {}

for name in ("gcn", "sage", "gat"):
    print(f"=== {name.upper()} ===")
    r = train_node_multi_seed(build(name), cora.to(DEVICE), model_name=name.upper(),
                              seeds=SEEDS, epochs=200, lr=1e-2, weight_decay=5e-4,
                              device=DEVICE)
    summary[name] = r
    histories[name.upper()] = r["results"][-1].history
    r["num_parameters"] = count_parameters(build(name)())
    print(f"  parameters : {r['num_parameters']:,}")
    print(f"  test acc   : {r['test_acc_mean']*100:.2f} +/- {r['test_acc_std']*100:.2f}%")
    print(f"  macro F1   : {r['test_f1_mean']*100:.2f} +/- {r['test_f1_std']*100:.2f}%")
    print(f"  baseline   : {r['baseline']*100:.2f}% (majority class on the test split)\n")

In [ ]:
viz.plot_training_curves(histories, title="Cora: GCN vs GraphSAGE vs GAT",
                         path=f"{RESULTS}/03_curves_all.png")

rows = [{"model": v["model"], "test_acc_mean": v["test_acc_mean"],
         "test_acc_std": v["test_acc_std"]} for v in summary.values()]
viz.plot_results_bars(rows, title="Cora test accuracy (mean over 3 seeds)",
                      baseline=summary["gcn"]["baseline"],
                      path=f"{RESULTS}/03_results.png")

In [ ]:
print(f"{'model':8}{'test acc':>20}{'macro F1':>20}{'params':>10}")
print("-" * 58)
for v in summary.values():
    print(f"{v['model']:8}{v['test_acc_mean']*100:>10.2f} +/- {v['test_acc_std']*100:>5.2f}%"
          f"{v['test_f1_mean']*100:>11.2f} +/- {v['test_f1_std']*100:>5.2f}%"
          f"{v['num_parameters']:>10,}")
print("-" * 58)
print(f"{'baseline':8}{summary['gcn']['baseline']*100:>10.2f}%  (majority class)")
print()
print("Published reference points for the same protocol:")
print("  GCN (Kipf & Welling 2017)   81.5%")
print("  GraphSAGE (Hamilton 2017)   ~81.6%")
print("  GAT (Velickovic 2018)       83.0% +/- 0.5%")
print("Landing within a point or two of these means the implementations are right.")

## 6. Reading the result critically

### Accuracy is not the whole story

Cora's test split is imbalanced. Macro-F1 averages per-class F1 *unweighted*,
so it tells you whether the model handles the rare classes too -- or whether it
is coasting on `Neural_Networks` (the largest class).

In [ ]:
from gnn_lab.metrics import per_class_report, confusion_matrix

set_seed(0)
model = build("gcn")().to(DEVICE)
train_node(model, cora.to(DEVICE), model_name="GCN", epochs=200, device=DEVICE)
model.eval()
with torch.no_grad():
    pred = model(cora.x, cora.edge_index).argmax(-1)

tm = cora.test_mask
rep = per_class_report(pred[tm].cpu(), cora.y[tm].cpu(), cora.num_classes,
                       cora.class_names or None)
print(f"{'class':<24}{'prec':>8}{'recall':>8}{'f1':>8}{'support':>9}")
print("-" * 57)
for i, nm in enumerate(rep["name"]):
    print(f"{nm:<24}{rep['precision'][i]:>8.3f}{rep['recall'][i]:>8.3f}"
          f"{rep['f1'][i]:>8.3f}{rep['support'][i]:>9}")
print("-" * 57)
print(f"{'macro average':<24}{'':>8}{'':>8}{sum(rep['f1'])/len(rep['f1']):>8.3f}"
      f"{sum(rep['support']):>9}")
print(f"\naccuracy: {accuracy(pred[tm].cpu(), cora.y[tm].cpu()):.4f}")

viz.plot_confusion(confusion_matrix(pred[tm].cpu(), cora.y[tm].cpu(), cora.num_classes),
                   class_names=rep["name"], title="GCN on Cora (test split, row-normalised)",
                   path=f"{RESULTS}/03_confusion.png")

### What the embeddings look like

t-SNE of the learned node embeddings. Classes form visible clusters -- which is
the point of the representation, and the thing that over-smoothing destroys
(notebook 04).

In [ ]:
with torch.no_grad():
    h = model(cora.x, cora.edge_index)
viz.plot_tsne(h.cpu(), cora.y.cpu(), title="GCN output logits, t-SNE",
              path=f"{RESULTS}/03_tsne.png")

### How much are those 140 labels worth?

The Planetoid protocol gives 20 labels per class. What happens with far more
supervision? This is the question that separates "GNNs are good at
semi-supervised learning" from "GNNs are just classifiers with extra steps".

In [ ]:
print(f"{'split':<28}{'train nodes':>12}{'test acc':>12}")
print("-" * 52)
for split in ("public", "random"):
    d = load_cora(source="linqs", row_normalize=False, split=split, verbose=False).to(DEVICE)
    set_seed(0)
    r = train_node(build("gcn")(), d, model_name="GCN", epochs=200, device=DEVICE)
    label = f"linqs/{split}" + (" (140 labels)" if split == "public" else " (60% labels)")
    print(f"{label:<28}{int(d.train_mask.sum()):>12,}{r.test_acc_at_best_val*100:>11.2f}%")
print()
print("The gap between those two rows is the value of the graph structure itself.")
print("With 60% of nodes labelled, an MLP on features alone does nearly as well --")
print("which is why the 140-label regime is the interesting one.")

## 7. Summary

* Cora: 2708 nodes, 1433 features, 7 classes, 10556 directed edges after
  de-duplication. Split 140/500/1000.
* Two loaders, same graph, different node orderings. The Planetoid one is
  literature-comparable.
* **Two data gotchas**: unsorted `test.index` needs an un-permutation, and the
  raw adjacency dict has 302 duplicate entries that inflate degrees.
* Report test accuracy **at the best validation epoch**, over **multiple
  seeds**, next to the **majority baseline** and **macro-F1**.
* Expected ballpark: GCN ~81%, SAGE ~81%, GAT ~83%.

**Next:** `04_equivalence_and_ablations.ipynb` -- prove the hand-written layers
match PyTorch Geometric numerically, then deliberately break them to find out
what each design choice is worth.